# 06 — Iteration 3: BM25 length normalization

**Hypothesis:** reducing length normalization may change rankings when short final chunks compete with longer passages. All baseline settings remain fixed except `b`, reduced from 0.75 to 0.30. Most chunks have equal length, so this parameter may have limited impact.

The train-only loop selected this untried proposal from its existing catalog. This notebook reviews the saved run and lets you try the candidate; running it does not advance the loop. The baseline and rejected candidate remain available.

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from improve import read_log, read_json
from retrieval import BM25Retriever, load_inputs

runs = [r for r in read_log(ROOT / 'experiments/loop_log.jsonl') if r['iteration'] == 3]
assert runs, 'Iteration 3 has not been recorded.'
run = runs[-1]
output = ROOT / run['result_directory']
print('Changed settings:', run['changes'])
print('Why:', run['proposal']['reason'])
print('Saved run:', output)
print('Accepted:', run['decision']['accepted'])

Changed settings: {'b': {'before': 0.75, 'after': 0.3}}
Why: Less length normalization may reduce ranking bias toward short fragments.
Saved run: C:\Users\vatsa\OneDrive\Desktop\JioStar\results\iteration_03\20261101T023005238322Z
Accepted: False


## Compare the same train questions

Use 140 answerable train questions and 23 frozen train negatives. Unique-character metrics count overlapping positions once; reference metrics follow the authors' pairwise scoring convention. Neither is a generated-answer accuracy score.

The acceptance rule was set before measuring the candidate. A positive gain can still be rejected if it is below the chosen minimum.

In [2]:
rows = []
for label, metrics in run['metrics'].items():
    at5 = next(m for m in metrics['retrieval'] if m['k'] == 5)
    rows.append(dict(approach=label, precision_at_5=at5['precision'], recall_at_5=at5['recall'],
                     reference_precision_at_5=at5['benchmark_precision'],
                     reference_recall_at_5=at5['benchmark_recall'],
                     false_abstention=metrics['false_abstention'],
                     correct_abstention=metrics['correct_abstention'], p95_ms=metrics['latency_ms']['p95']))
comparison = pd.DataFrame(rows)
display(comparison)
reference_recall = comparison.loc[comparison['approach'] == 'reference', 'recall_at_5'].iloc[0]
candidate_recall = comparison.loc[comparison['approach'] == 'candidate', 'recall_at_5'].iloc[0]
print(f'Recall gain: {(candidate_recall-reference_recall)*100:.2f} percentage points')
print(f"Required gain: {run['objective']['minimum_recall_gain_absolute']*100:.2f} percentage points")
display(pd.Series(run['decision']['checks'], name='Passed'))
comparison.to_csv(output / 'comparison.csv', index=False)

,approach,precision_at_5,recall_at_5,reference_precision_at_5,reference_recall_at_5,false_abstention,correct_abstention,p95_ms
0,baseline,0.017433,0.181647,0.017594,0.192427,0.0,0.0,11.648065
1,reference,0.017433,0.181647,0.017594,0.192427,0.0,0.0,11.648065
2,candidate,0.017916,0.188202,0.018035,0.198982,0.0,0.0,10.469325


Recall gain: 0.66 percentage points
Required gain: 1.00 percentage points


recall_gain                False
precision                   True
answerable_abstention       True
unanswerable_abstention     True
latency                     True
Name: Passed, dtype: bool

## Inspect individual changes

Macro averages can hide regressions. Compare each saved train question at k=5 to identify which questions gained or lost coverage. These are diagnostics, not additional proposal trials.

In [3]:
reference = read_json(output / 'reference.json')
candidate = read_json(output / 'candidate.json')
a = pd.DataFrame(reference['answerable']['per_query'])
b = pd.DataFrame(candidate['answerable']['per_query'])
changes = a[a['k'] == 5][['query_id','recall']].merge(
    b[b['k'] == 5][['query_id','recall']], on='query_id', suffixes=('_before','_after'))
changes['recall_delta'] = changes['recall_after'] - changes['recall_before']
questions = pd.DataFrame(reference['answerable']['predictions'])[['query_id','query']]
changed = changes[changes['recall_delta'] != 0].merge(questions, on='query_id').sort_values('recall_delta')
display(changed)
print('Questions improved:', (changes['recall_delta'] > 0).sum())
print('Questions worsened:', (changes['recall_delta'] < 0).sum())

,query_id,recall_before,recall_after,recall_delta,query
0,c0283a17faa7ef1431c129e18744f28d6465c4484a67d5...,0.0,0.917722,0.917722,Consider the Walabot HOME Reseller Agreement b...


Questions improved: 1
Questions worsened: 0


## Try the candidate

Edit `question` or choose `reference_config` to run the original baseline. Settings are loaded from this run's preserved configuration.

In [4]:
which = 'candidate_config'
_, documents, _ = load_inputs(ROOT)
model = BM25Retriever(documents, run[which])
question = 'Under the Fox and Glu Mobile wireless content license, when does the term expire?'
hits = model.search(question, k=5)
if hits:
    display(pd.DataFrame(hits)[['score','file_path','start','end','text']])
else:
    print('No relevant passage found.')

,score,file_path,start,end,text
0,53.591533,cuad/GluMobileInc_20070319_S-1A_EX-10.09_43663...,0,1000,"AMENDMENT NO. 3\n\nDated as of February 19, 20..."
1,30.920385,cuad/GluMobileInc_20070319_S-1A_EX-10.09_43663...,800,1800,"dom of Heaven, Mr. and Mrs. Smith, In Her Shoe..."
2,23.672805,cuad/GluMobileInc_20070319_S-1A_EX-10.09_43663...,1600,1902,"x Mobile Entertainment, Inc. (""Fox"")\n\n\n\nBy..."
3,20.484946,cuad/FLOTEKINDUSTRIESINCCN_05_09_2019-EX-10.1-...,1600,2600,"ting Principles, consistently applied), plus a..."
4,20.341584,cuad/KINGPHARMACEUTICALSINC_08_09_2006-EX-10.1...,65600,66600,"grants to Depomed a non-assignable, non-subli..."


## Explain this step

“We reduced the BM25 parameter that adjusts for passage length. Train recall@5 improved from 18.16% to 18.82%, and precision from 1.74% to 1.79%. The recall gain was only 0.66 percentage points, below our predeclared 1-point requirement, so we kept the baseline.”

This threshold is an engineering choice, not proof of statistical significance. We preserve the candidate as a small positive result. Both configurations still return passages for all 23 train negatives; this parameter change does not solve answerability. The model remains a development prototype.

**Progress:** three distinct iterations beyond baseline are complete; at least two more are required. No test performance has been measured. We stop here before the next proposal.